## - Generate the 70/30 sampling from all CCEs

In [12]:
import pandas as pd
import random
from pathlib import Path

# =========================
# CONFIG (V3.1-aligned)
# =========================
INPUT_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0")
INPUT_CSV  = INPUT_DIR / "episodes_enriched_combined.csv"   # list of all CCEs (V3.1)

OUTPUT_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#SAMPLE_SIZE        = 1000    # total size across DEV+TEST - Original Size
SAMPLE_SIZE        = 300    # total size across DEV+TEST
#SEED               = 12345   # fixed seed for reproducibility - Original Seed
SEED               = 1234567   # fixed seed for reproducibility
STRATIFY_BY_REPO   = False   # True = proportional per repo
#DEV_RATIO          = 0.70    # 70% to DEV, 30% to TEST - Original Ratio
DEV_RATIO          = 0.0    # 0% to DEV, 100% to TEST

# =========================
# HELPERS
# =========================
def sample_rows(df_in: pd.DataFrame, n: int, seed: int, stratify: bool) -> pd.DataFrame:
    """Sample n rows from df_in (optionally stratified by repo)."""
    if df_in.empty:
        return df_in
    n = min(n, len(df_in))
    if not stratify or "repo" not in df_in.columns:
        return df_in.sample(n=n, random_state=seed)

    # Proportional by repo
    rng = random.Random(seed)
    parts = []
    total = len(df_in)
    for repo, g in df_in.groupby("repo", sort=False):
        k = max(1, round(n * (len(g) / total)))
        parts.append(g.sample(n=min(k, len(g)), random_state=rng.randint(0, 10**9)))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)  # trim to exact n
    return out

# =========================
# MAIN
# =========================
def main():
    # 1) Load all CCEs
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    if df.empty:
        raise ValueError("Input CCE list is empty.")
    print(f"[INFO] Loaded CCEs: {len(df):,} rows from {INPUT_CSV}")

    # Preserve original column order to enforce identical schema
    original_cols = df.columns.tolist()

    # 2) Draw overall sample of CCE rows (no column changes)
    sample_all = sample_rows(df, SAMPLE_SIZE, SEED, STRATIFY_BY_REPO)

    # 3) Split into DEV/TEST (70/30) with fixed seed
    sample_all = sample_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle once
    n_total = len(sample_all)
    n_dev = int(round(n_total * DEV_RATIO))
    n_test = n_total - n_dev

    dev_df  = sample_all.iloc[:n_dev].copy().reindex(columns=original_cols)
    test_df = sample_all.iloc[n_dev:].copy().reindex(columns=original_cols)

    # 4) Save full columns, identical to input (order preserved)
    dev_path  = OUTPUT_DIR / "DEV_Commit_Sample.csv"
    test_path = OUTPUT_DIR / "TEST_Commit_Sample.csv"

    dev_df.to_csv(dev_path, index=False, encoding="utf-8")
    test_df.to_csv(test_path, index=False, encoding="utf-8")

    # Optional sanity checks
    assert dev_df.columns.tolist()  == original_cols, "DEV columns differ from input!"
    assert test_df.columns.tolist() == original_cols, "TEST columns differ from input!"

    print(f"[OK] DEV written : {dev_path}  (rows={len(dev_df):,}, cols={len(original_cols)})")
    print(f"[OK] TEST written: {test_path} (rows={len(test_df):,}, cols={len(original_cols)})")
    print(f"[OK] Total sampled: {n_total:,} (DEV={len(dev_df):,}, TEST={len(test_df):,})")

if __name__ == "__main__":
    main()


[INFO] Loaded CCEs: 12,700 rows from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\episodes_enriched_combined.csv
[OK] DEV written : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\DEV_Commit_Sample.csv  (rows=0, cols=37)
[OK] TEST written: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\TEST_Commit_Sample.csv (rows=300, cols=37)
[OK] Total sampled: 300 (DEV=0, TEST=300)


## 3 - Aggregate the manual review results: Detection_List

In [5]:
# -*- coding: utf-8 -*-
"""
Build Detection_List.csv: one row per intent with comma-separated,
alphabetically sorted, de-duplicated detected keywords.

Input folder (Windows):
C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent
"""

from pathlib import Path
import pandas as pd
import csv
import sys

# --- Configure your folder path here ---
FOLDER = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent")

def read_csv_any(path: Path) -> pd.DataFrame:
    """Read CSV with sensible encodings fallback."""
    for enc in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    # Last attempt without encoding param to surface a clear error
    return pd.read_csv(path)

def pick_input_csv(folder: Path) -> Path:
    """Pick a CSV in the folder. If multiple, prefer ones that look like labeled outputs."""
    candidates = sorted(folder.glob("*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No CSV files found in: {folder}")
    # Prefer files that likely contain the labeled sample
    preferred_names = ("DEV_Commit_Sample_Labeled.csv",
                       "Main_Commit_List_Labeled_Intents.csv",
                       "Main_Commit_List_Labeled.csv")
    for name in preferred_names:
        p = folder / name
        if p.exists():
            return p
    # Fall back to first CSV
    return candidates[0]

def parse_detected_keywords_field(s: str):
    """
    Parse 'detected_keywords' formatted like:
        'label: kw1|kw2; label2: kw3|kw4'
    Returns dict[label] -> set(keywords)
    """
    out = {}
    if not isinstance(s, str) or not s.strip():
        return out
    # Split by ';' for label segments
    segments = [seg.strip() for seg in s.split(";") if seg.strip()]
    for seg in segments:
        if ":" not in seg:
            continue
        label, kws_part = seg.split(":", 1)
        label = label.strip()
        kws = [kw.strip().lower() for kw in kws_part.split("|") if kw.strip()]
        if not kws:
            continue
        out.setdefault(label, set()).update(kws)
    return out

def aggregate_keywords(df: pd.DataFrame):
    """
    Aggregate keywords by intent label using either:
      - 'intent' + 'detected_keywords' columns
      - 'intent_label' + 'basis_keyword' columns
    Returns dict[label] -> set(keywords)
    """
    agg = {}

    has_detected = ("intent" in df.columns) and ("detected_keywords" in df.columns)
    has_pairwise = ("intent_label" in df.columns) and ("basis_keyword" in df.columns)

    if not has_detected and not has_pairwise:
        raise ValueError("Input CSV must contain either "
                         "['intent','detected_keywords'] or ['intent_label','basis_keyword'] columns.")

    if has_detected:
        for val in df["detected_keywords"].astype(str):
            per_label = parse_detected_keywords_field(val)
            for label, kws in per_label.items():
                agg.setdefault(label, set()).update(kws)

    if has_pairwise:
        # Each row has a single label with a single basis keyword
        for label, kw in zip(df["intent_label"].astype(str), df["basis_keyword"].astype(str)):
            label = label.strip()
            kw = kw.strip().lower()
            if not label or not kw:
                continue
            agg.setdefault(label, set()).add(kw)

    return agg

def main():
    in_csv = pick_input_csv(FOLDER)
    df = read_csv_any(in_csv)

    agg = aggregate_keywords(df)

    # Normalize labels to lower-case for consistency, then build rows
    rows = []
    for label in sorted(agg.keys(), key=lambda x: x.lower()):
        # sort keywords alphabetically
        kws_sorted = sorted(agg[label])
        rows.append({
            "intent_label": label,
            "keywords": ", ".join(kws_sorted)
        })

    out_csv = FOLDER / "Detection_List.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
    print(f"Wrote: {out_csv}")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Error: {e}", file=sys.stderr)
        sys.exit(1)


Wrote: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent\Detection_List.csv


## 3 - Review the Sample with the detection list - assign the labels to the the sample set

In [18]:
import re
from pathlib import Path
import pandas as pd

# =========================================
# CONFIG — change these paths for your machine
# =========================================
BASE_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent")

DETECTION_CSV = BASE_DIR / "Detection_List_V3.0.csv"      # detection list with columns: intent_label, regex, keywords
DEV_COMMITS   = BASE_DIR / "TEST_Commit_Sample.csv"         # commits to label
OUT_CSV       = BASE_DIR / "TEST_Commit_Sample_labeled_V3.0.csv"

# If your CSVs are UTF-8 with BOM or plain UTF-8, these helpers will handle both.
READ_KWARGS_PRIMARY   = dict(encoding="utf-8-sig")
READ_KWARGS_FALLBACK  = dict(encoding="utf-8", errors="ignore")

# =========================================
# HELPERS
# =========================================
PREFERRED_TEXT_COLS = [
    "subject_norm", "subject_raw", "subject", "message", "title",
    "commit", "summary", "desc", "description"
]

def pick_text_column(df: pd.DataFrame) -> str:
    """Pick the best commit subject column."""
    for c in PREFERRED_TEXT_COLS:
        if c in df.columns:
            return c
    for c in df.columns:
        if df[c].dtype == "object":
            return c
    # last resort
    return df.columns[0]

def load_detection_list(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, **READ_KWARGS_PRIMARY)
    except UnicodeDecodeError:
        return pd.read_csv(path, **READ_KWARGS_FALLBACK)

def load_commits(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, **READ_KWARGS_PRIMARY)
    except UnicodeDecodeError:
        return pd.read_csv(path, **READ_KWARGS_FALLBACK)

def compile_detection_patterns(det_df: pd.DataFrame):
    """
    Build a list of dicts: {"intent": str, "regex": compiled_re, "keywords": [list]}
    - Columns may be cased differently; handle case-insensitively.
    - 'keywords' can be comma/semicolon/pipe separated.
    - Invalid regex rows are skipped with a warning.
    """
    col_map = {c.lower(): c for c in det_df.columns}
    required = {"intent_label", "regex", "keywords"}
    if not required.issubset(set(col_map.keys())):
        raise ValueError(
            f"Detection list must include columns: {', '.join(sorted(required))}. "
            f"Got: {', '.join(det_df.columns)}"
        )

    intents_col  = col_map["intent_label"]
    regex_col    = col_map["regex"]
    keywords_col = col_map["keywords"]

    patterns = []
    for idx, row in det_df.iterrows():
        intent = str(row[intents_col]).strip()
        rx_str = str(row[regex_col]).strip()
        kw_raw = str(row.get(keywords_col, "")).strip()

        if not intent or not rx_str:
            continue

        try:
            # External case-insensitive flag; inline flags inside rx_str are fine too.
            rx = re.compile(rx_str, re.I)
        except re.error as e:
            print(f"[WARN] Skipping invalid regex for intent '{intent}' (row {idx+1}): {e}")
            continue

        kw_list = [k.strip() for k in re.split(r"[|;,]", kw_raw) if k.strip()]
        patterns.append({"intent": intent, "regex": rx, "keywords": kw_list})

    if not patterns:
        raise ValueError("No usable patterns found in detection list.")
    return patterns

def detect_for_subject(subject: str, patterns):
    """
    Return (intents_csv, keywords_csv) for this subject.
    - intents_csv: all intent labels whose regex matched.
    - keywords_csv: all keywords (from the detection list for matched intents) that appear in the subject.
      Uses word-boundary match for single tokens; substring match for phrases (space, slash, dash, underscore).
    Both outputs are sorted (case-insensitive), de-duplicated, and comma-separated.
    """
    s = "" if pd.isna(subject) else str(subject)
    intents_hit = []
    matched_keywords = []

    for entry in patterns:
        if entry["regex"].search(s):
            intents_hit.append(entry["intent"])
            for kw in entry["keywords"]:
                # For phrases or tokens with separators, use a relaxed substring match (case-insensitive).
                if any(ch in kw for ch in (" ", "/", "-", "_")):
                    kw_rx = re.compile(re.escape(kw), re.I)
                else:
                    # Single tokens: use word boundary to avoid partial word matches.
                    kw_rx = re.compile(rf"\b{re.escape(kw)}\b", re.I)
                if kw_rx.search(s):
                    matched_keywords.append(kw.lower())

    intents_csv  = ", ".join(sorted(set(intents_hit), key=str.lower))
    keywords_csv = ", ".join(sorted(set(matched_keywords)))
    return intents_csv, keywords_csv

# =========================================
# MAIN
# =========================================
def main():
    if not DETECTION_CSV.exists():
        raise FileNotFoundError(f"Detection list not found: {DETECTION_CSV}")
    if not DEV_COMMITS.exists():
        raise FileNotFoundError(f"Commit sample not found: {DEV_COMMITS}")

    det_df = load_detection_list(DETECTION_CSV)
    patterns = compile_detection_patterns(det_df)

    commits_df = load_commits(DEV_COMMITS)
    text_col = pick_text_column(commits_df)

    results = commits_df[text_col].astype(str).apply(lambda s: detect_for_subject(s, patterns))
    commits_df["Intent"]   = results.apply(lambda x: x[0])
    commits_df["Keywords"] = results.apply(lambda x: x[1])

    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    commits_df.to_csv(OUT_CSV, index=False, encoding="utf-8")
    print(f"[OK] Labeled file written: {OUT_CSV}")

    # Optional: quick summary
    if "Intent" in commits_df.columns:
        counts = (
            commits_df["Intent"]
            .str.split(r"\s*,\s*")
            .explode()
            .dropna()
            .loc[lambda s: s.str.len() > 0]
            .value_counts()
        )
        if not counts.empty:
            print("\nPredicted counts by intent:")
            for intent, n in counts.items():
                print(f"  {intent}: {n}")

if __name__ == "__main__":
    main()


[WARN] Skipping invalid regex for intent 'speed_up' (row 6): unbalanced parenthesis at position 186
[OK] Labeled file written: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent\TEST_Commit_Sample_labeled_V3.0.csv

Predicted counts by intent:
  update/upgrade: 110
  version_change: 61
  release&tagging: 47
  fail_fix: 37
  build/infra: 30
  Removal: 12
  chore: 12
  CI_config: 12
  cleanup: 8
  doc/chonelog: 6
  revert: 1
